# 🍄 25 · From maps to biology: membranes, drugs, fitting atoms and the proteome in situ
### Turning particle coordinates and density maps into biological claims — and knowing how strong each claim is

**BiologyToResearchV3** · Part F — Frontiers, biology, research practice · Module 25 of 26

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearchV3/notebooks/25_from_maps_to_biology.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`spatial`, `cluster`, `stats`), scikit-learn, Matplotlib |
| **Prerequisites** | Modules 04 (statistics, GMM/BIC), 09 (validation), 10–11 (cross-correlation search), 14 (pattern mining) |
| **Estimated time** | 140 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | **Jiang *et al.* 2025** (Nat. Commun. — fungal plasma membrane); **Zhang, Vasishtan, Xu, Topf & Alber 2010** (Bioinformatics — simultaneous fitting); **Beck *et al.* 2011** (J. Struct. Biol. — proteome organisation, review); **Thalassinos *et al.* 2013** (Structure — perspective); biological findings of **Xu *et al.* 2019** (MPP) and **Zeng *et al.* 2023** (DISCA) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The membrane point patterns, the toy assembly and the in silico bacterium are **simulated**; their parameters are
> inspired by the papers but the data are not the papers' data. Where we build an effect into a simulation (for example a
> 1.7× cluster expansion), we say so — the point is to test whether an analysis can *recover* it.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Quantify** the organisation of membrane proteins from particle coordinates: nearest-neighbour distances, rule-based
   clusters, cluster radii and Gaussian-mixture checks.
2. **Test** differences between conditions with Kruskal–Wallis and Dunn tests, choosing the right **unit of
   replication**, and report effect sizes with confidence intervals.
3. **Explain** why fitting components into a map one at a time fails, and **implement** simultaneous fitting with a
   no-overlap constraint.
4. **Convert** particle counts into concentrations and **compare** spatial patterns against a well-chosen null model
   (depletion tests, pair-correlation functions).
5. **Simulate** diffusion in a crowded cytoplasm and explain why crowding matters for biology.
6. **Grade** a biological claim on a ladder of evidence: observation → statistics → perturbation → orthogonal
   measurement → mechanism.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · From numbers to claims | the ladder of evidence | 5 min |
| 2 · Case study: the fungal plasma membrane | point patterns, clusters, statistics, effect sizes | 45 min |
| 3 · Fitting components into a map | exhaustive search, the copy problem, simultaneous fitting | 30 min |
| 4 · The proteome in situ | concentrations, spatial statistics, crowding and diffusion | 35 min |
| 5 · What the lab's pattern mining found | MPP and DISCA as biology | 10 min |
| 📄 / ✅ / 🧠 / 🚀 | paper connections, checkpoint, discussion, going further | 15 min |

## 🧫 The biological question

*Candida glabrata* is a yeast that causes serious, often drug-resistant infections. Echinocandin drugs such as
**caspofungin** block β-(1,3)-glucan synthase (GS, the protein Fks1), which builds the cell wall from inside the plasma
membrane. GS does not work alone: it sits in a membrane crowded with other large complexes, especially the proton pump
**Pma1**, which forms hexamers about 16 nm across. Jiang *et al.* (2025, with the Xu lab) imaged isolated fungal plasma
membranes by cryo-ET and asked: *how are these complexes arranged, and how does the drug change the arrangement?*
Answering needs more than pictures. It needs coordinates, measurements, a null hypothesis, statistics, and orthogonal
experiments. This module is about that last mile — from maps to biology.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — fine, everything runs on a CPU.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial import cKDTree
from sklearn.mixture import GaussianMixture

SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import struct
import urllib.request

POLNET_COMMIT = "73e613cc9162d26c7fd740ce3299bcb280a3e3b6"
POLNET_URL = ("https://raw.githubusercontent.com/anmartinezs/polnet/"
              + POLNET_COMMIT + "/data/templates/{kind}/{name}.{ext}")
CACHE_DIR = os.environ.get("CRYO_COURSE_CACHE", "course_data")
ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2277, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=5851, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3325, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=659, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=839, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1309, symmetry="C1"),
    # NB: PolNet's 4V94 map holds TWO TRiC complexes side by side (1.95 MDa ≈ 2 × ~1 MDa).
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1952, symmetry="C2"),
    "1bxn": dict(name="RuBisCO", organism="Cupriavidus necator", mass_kDa=280, symmetry="D4"),
    "1qvr": dict(name="ClpB chaperone", organism="Thermus thermophilus", mass_kDa=297, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=None, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=None, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=239, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=236, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=43, symmetry="C1"),
}
def _download(url: str, dest: str, timeout: float = 60.0) -> bool:
    """Download ``url`` to ``dest`` (no-op if it exists). Returns True on success."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return True
    except Exception as exc:  # network blocked, offline, ...
        print(f"  ⚠️  download failed for {url.split('/')[-1]}: {type(exc).__name__}")
        return False


def read_mrc(path_or_bytes) -> tuple[np.ndarray, float]:
    """Minimal MRC2014 reader (modes 0, 1, 2, 6). Returns (array[z, y, x], voxel_size_Å).

    The MRC header is 1024 bytes of little-endian 32-bit words; words 1–3 are the
    grid size (nx, ny, nz), word 4 the data type, words 11–13 the cell size in Å,
    and word 24 the length of an optional extended header.
    """
    raw = path_or_bytes if isinstance(path_or_bytes, (bytes, bytearray)) else open(path_or_bytes, "rb").read()
    if raw[:2] == b"\x1f\x8b":  # gzip
        import gzip
        raw = gzip.decompress(raw)
    nx, ny, nz, mode = struct.unpack("<4i", raw[0:16])
    mx, my, mz = struct.unpack("<3i", raw[28:40])
    cella = struct.unpack("<3f", raw[40:52])
    mapc, mapr, maps = struct.unpack("<3i", raw[64:76])
    nsymbt = struct.unpack("<i", raw[92:96])[0]
    dtype = {0: np.int8, 1: np.int16, 2: np.float32, 6: np.uint16}[mode]
    data = np.frombuffer(raw, dtype=dtype, offset=1024 + nsymbt, count=nx * ny * nz)
    vol = data.reshape(nz, ny, nx).astype(np.float32)
    if (mapc, mapr, maps) != (1, 2, 3) and (mapc, mapr, maps) != (0, 0, 0):
        # reorder axes so that the array is [z, y, x]
        order = [3 - maps, 3 - mapr, 3 - mapc]
        vol = np.transpose(vol, np.argsort(order))
    voxel = float(cella[0] / mx) if mx > 0 else 1.0
    return vol, voxel


def fetch_zoo_map(pdb_id: str, voxel: str = "10A") -> tuple[np.ndarray, float, str]:
    """Return (density[z, y, x], voxel_size_Å, source) for a molecular-zoo entry.

    Tries the pinned PolNet template first; if the download fails (no internet),
    falls back to a *procedural* stand-in so the notebook still runs.
    """
    pdb_id = pdb_id.lower()
    dest = os.path.join(CACHE_DIR, f"polnet_{voxel}_{pdb_id}.mrc")
    url = POLNET_URL.format(kind=f"mrcs_{voxel}", name=pdb_id, ext="mrc")
    if _download(url, dest):
        vol, vs = read_mrc(dest)
        return vol, vs, f"PolNet template from PDB {pdb_id.upper()}"
    vol = procedural_molecule(pdb_id)
    return vol, 10.0, f"procedural stand-in for {pdb_id.upper()} (offline fallback)"


def procedural_molecule(pdb_id: str, size: int = 40) -> np.ndarray:
    """Offline fallback: a coarse shape with the right size/symmetry flavour (1 nm voxels)."""
    g = (np.arange(size) - size / 2 + 0.5)
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    r = np.sqrt(x**2 + y**2)

    def blob(cz, cy, cx, s):
        return np.exp(-((z - cz)**2 + (y - cy)**2 + (x - cx)**2) / (2 * s**2))
    pid = pdb_id.lower()
    if pid in ("4v4r", "4v7r", "5mrc"):          # two unequal subunits
        v = blob(-3, 0, 0, 7.5) + 0.8 * blob(6, 2, 1, 5.5)
    elif pid in ("3j9i", "6utj", "4cr2"):        # stacked rings / barrel
        v = np.exp(-((r - 5.5)**2) / 4.0) * (np.abs(z) < 7.5)
        if pid != "3j9i":
            v = v + 0.8 * blob(10, 0, 0, 4)
    elif pid == "4v94":                          # double ring with cavity
        v = np.exp(-((r - 7)**2) / 6.0) * (np.abs(z) < 8)
    elif pid in ("3cf3", "1qvr"):                # hexameric ring
        ang = np.arctan2(y, x)
        v = np.exp(-((r - 5)**2) / 3.0) * (np.abs(z) < 4) * (1 + 0.5 * np.cos(6 * ang))
    elif pid == "1bxn":                          # compact ball
        v = blob(0, 0, 0, 4.5)
    else:                                        # small asymmetric blob
        v = blob(0, 0, 0, 3) + 0.5 * blob(2, 2, 0, 2)
    return ndimage.gaussian_filter(v.astype(np.float32), 1.0)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    PolNet templates are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    com = np.array(ndimage.center_of_mass(v))
    shift = np.array(v.shape) / 2 - 0.5 - com
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)
    dst0 = np.maximum((size - s) // 2, 0)
    n = np.minimum(s, size)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
print("🧰 toolkit loaded")

---
## 1 · From numbers to claims: the ladder of evidence

📍 **Where we are.** Modules 10–24 turned tomograms into *particles*: coordinates, orientations, classes and averages.
A paper, however, makes *biological* claims. It helps to grade every claim by the kind of evidence behind it:

| Rung | Evidence | Fungal-membrane example | Typical tool |
|---|---|---|---|
| 1 · Observation | what is there, how many, where | Pma1 hexamers form dense clusters | picking, classification, averaging |
| 2 · Statistics | the pattern is unlikely under a null model | clusters are tighter than random placement would give | point-pattern statistics, tests |
| 3 · Perturbation | changing a cause changes the pattern | caspofungin spreads the clusters apart | treated vs untreated, mutants |
| 4 · Orthogonal measurement | an independent method agrees | the drug also lowers membrane tension | micropipette aspiration, proteomics |
| 5 · Mechanism | a model explains all of the above | drug binding to GS perturbs lipids, which reorganise proteins | modelling, further experiments |

Computation lives mostly on rungs 1–2. Good papers say clearly which rung each sentence stands on. Keep this table in
mind for the rest of the module.

---
## 2 · Case study: the molecular landscape of the fungal plasma membrane

**What Jiang *et al.* (2025) did.** They isolated plasma membranes from *C. glabrata* spheroplasts and recorded about 100
tomograms (14 high-contrast ones were analysed), plus tomograms of FIB-milled lamellae of whole cells. Pma1 hexamers,
GS, ribosomes and glycogen were annotated with EMAN2's convolutional-network tool (one small network per feature) and
curated by hand. They then measured **nearest-neighbour distances**, defined **clusters** as at least four hexamers at
about 160–170 Å spacing, checked them with **Gaussian mixture models** (AIC/BIC) and **k-means**, and compared conditions
with **Kruskal–Wallis** and **Dunn** tests.

**Our toy.** We simulate the *output* of the annotation step — hexamer centres in 1 µm × 1 µm membrane patches — for
three conditions, seven patches each:

* **wild type (WT)**: about 264 hexamers per µm² (the density reported in the paper), 80 % of them packed in clusters on a
  jittered hexagonal lattice with 16.5 nm spacing, the rest isolated;
* **WT + caspofungin**: the same, but we **build in** a 1.7× expansion of the spacing (the size of the effect the paper
  reports for wild type);
* **Δ*fen1*** (a sphingolipid mutant whose Pma1 the paper found already dispersed): a 1.5× expansion — **an invented
  number** for the toy.

Each patch also gets its own small random spacing factor (±6 %), because real membranes, cells and tomograms differ.
This matters for the statistics in 2.3.

In [ ]:
# ── 2.1 Simulated hexamer centres in membrane patches ──
L_PATCH = 1000.0                                                   # patch side (nm)
_i, _j = np.meshgrid(np.arange(-12, 13), np.arange(-12, 13))
HEX_LATTICE = np.stack([_i + 0.5 * _j, _j * np.sqrt(3) / 2], -1).reshape(-1, 2)    # unit-spacing hexagonal lattice


def membrane_patch(rng, spacing=16.5, jitter=1.5, density=264, frac_clustered=0.8, mean_size=18, patch_sd=0.06):
    """Hexamer centres (nm) in one patch: compact lattice clusters + isolated hexamers. Returns (points, cluster label)."""
    f = rng.lognormal(0, patch_sd)                                  # patch-to-patch biological variability
    spacing, jitter = spacing * f, jitter * f
    n_total = rng.poisson(density * (L_PATCH / 1000) ** 2)
    n_clustered = int(frac_clustered * n_total)
    clusters, discs = [], []
    while sum(len(c) for c in clusters) < n_clustered:
        n = int(np.clip(rng.lognormal(np.log(mean_size), 0.5), 4, max(4, n_clustered - sum(len(c) for c in clusters))))
        a = rng.uniform(0, np.pi / 3)
        rot = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
        lat = HEX_LATTICE * spacing
        order = np.argsort(np.linalg.norm(lat, axis=1) + rng.uniform(0, 0.8 * spacing, len(lat)))
        shape = lat[order[:n]] @ rot.T                               # a compact, irregular patch of lattice
        rad = np.linalg.norm(shape, axis=1).max() + spacing
        for _ in range(200):
            c = rng.uniform(rad, L_PATCH - rad, 2)
            if all(np.linalg.norm(c - c2) > rad + r2 for c2, r2 in discs):
                break
        else:
            break                                                    # patch is full
        discs.append((c, rad))
        clusters.append(c + shape + rng.normal(0, jitter, shape.shape))
    pts = np.concatenate(clusters)
    labels = np.concatenate([[k] * len(c) for k, c in enumerate(clusters)])
    tree, isolated = cKDTree(pts), []
    while len(pts) + len(isolated) < n_total:
        c = rng.uniform(0, L_PATCH, 2)
        if tree.query(c)[0] > 3 * spacing and all(np.linalg.norm(c - o) > 3 * spacing for o in isolated):
            isolated.append(c)
    return np.vstack([pts, np.array(isolated).reshape(-1, 2)]), np.concatenate([labels, -np.ones(len(isolated), int)])


CONDITIONS = {"WT": 1.0, "WT + caspofungin": 1.7, "Δfen1": 1.5}         # spacing expansion built into the toy
N_PATCHES = 7
t0 = time.time()
patches = {c: [membrane_patch(rng, spacing=16.5 * k, jitter=1.5 * k) for _ in range(N_PATCHES)]
           for c, k in CONDITIONS.items()}
print(f"simulated {len(CONDITIONS) * N_PATCHES} patches in {time.time() - t0:.1f} s")
for c, ps in patches.items():
    print(f"{c:18s} hexamers per patch: {[len(p) for p, _ in ps]}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, (c, ps) in zip(axes, patches.items()):
    p, lab = ps[0]
    ax.scatter(p[:, 0], p[:, 1], s=4, c=np.where(lab >= 0, "tab:blue", "tab:gray"))
    ax.set_title(f"{c}  (1 µm² patch)"); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### 2.2 Nearest-neighbour distances

The simplest description of a point pattern: for every hexamer, the distance to its closest neighbour (NND). Tight
clusters give a sharp peak near the lattice spacing; dispersal shifts and broadens it.

In [ ]:
# ── 2.2 Nearest-neighbour distance distributions ──
def nnd(points):
    """Distance from every point to its nearest neighbour (nm)."""
    return cKDTree(points).query(points, k=2)[0][:, 1]


fig, ax = plt.subplots(figsize=(6.4, 3.2))
for c, ps in patches.items():
    d = np.concatenate([nnd(p) for p, _ in ps])
    ax.hist(d, bins=np.arange(0, 80, 1.5), histtype="step", lw=1.5, density=True, label=f"{c} (median {np.median(d):.1f} nm)")
ax.axvspan(16, 17, color="gold", alpha=0.4, label="160–170 Å")
ax.set_xlabel("nearest-neighbour distance (nm)"); ax.set_ylabel("density"); ax.legend(fontsize=8)
plt.show()

### 2.3 Statistics: what is the unit of replication?

The **Kruskal–Wallis test** asks whether several groups come from the same distribution, using only the **ranks** of the
values (robust to outliers and skew). Rank all $N$ values together; with $\bar R_g$ the mean rank of group $g$ ($n_g$ values),

$$H=\frac{12}{N(N+1)}\sum_g n_g\Big(\bar R_g-\frac{N+1}{2}\Big)^2 ,$$

which follows a $\chi^2$ distribution with (groups − 1) degrees of freedom when there is no difference. If $H$ is
significant, **Dunn's test** compares pairs of groups using the same ranks:

$$z_{ab}=\frac{\bar R_a-\bar R_b}{\sqrt{\frac{N(N+1)}{12}\left(\frac1{n_a}+\frac1{n_b}\right)}},$$

with the p-values multiplied by the number of comparisons (Bonferroni correction).

**But what is "a value"?** One option treats every hexamer as an independent observation (thousands of values). The
other summarises each **patch** by its median NND (seven values per condition). Hexamers in the same patch share that
patch's history, so they are not independent — treating them as such is called **pseudo-replication**.

In [ ]:
# ── 2.3a Kruskal–Wallis and Dunn tests ──
def dunn_test(groups, names):
    """Pairwise Dunn z-tests on pooled ranks with Bonferroni correction (no tie correction)."""
    allv = np.concatenate(groups)
    ranks = stats.rankdata(allv)
    N, out, start = len(allv), {}, 0
    mean_rank = []
    for g in groups:
        mean_rank.append(ranks[start:start + len(g)].mean()); start += len(g)
    pairs = list(itertools.combinations(range(len(groups)), 2))
    for a, b in pairs:
        z = (mean_rank[a] - mean_rank[b]) / np.sqrt(N * (N + 1) / 12 * (1 / len(groups[a]) + 1 / len(groups[b])))
        out[(names[a], names[b])] = min(1.0, 2 * stats.norm.sf(abs(z)) * len(pairs))
    return out


names = list(patches)
per_hexamer = [np.concatenate([nnd(p) for p, _ in patches[c]]) for c in names]
per_patch = [np.array([np.median(nnd(p)) for p, _ in patches[c]]) for c in names]
for label, groups in [("per hexamer", per_hexamer), ("per patch (median NND)", per_patch)]:
    H, p = stats.kruskal(*groups)
    print(f"Kruskal–Wallis, {label:24s}: n = {[len(g) for g in groups]}, H = {H:.1f}, p = {p:.2g}")
print("Dunn (per patch, Bonferroni):", {k: f"{v:.3g}" for k, v in dunn_test(per_patch, names).items()})

Both tests are "significant" here, because we built a large effect into the simulation. The danger of pseudo-replication
shows up when there is **no** effect. Below we simulate 40 WT patches, then 200 times pick 14 of them at random and call
7 "control" and 7 "treated". Any difference is pure chance, so a valid test at α = 0.05 should be "significant" in about
5 % of these fake experiments.

In [ ]:
# ── 2.3b A null experiment: how often does each analysis find a fake effect? ──
null_patches = [membrane_patch(rng)[0] for _ in range(40)]
null_nnd = [nnd(p) for p in null_patches]
hits_hexamer = hits_patch = 0
for _ in range(200):
    pick = rng.permutation(40)[:14]
    a, b = pick[:7], pick[7:]
    hits_hexamer += stats.mannwhitneyu(np.concatenate([null_nnd[i] for i in a]),
                                       np.concatenate([null_nnd[i] for i in b])).pvalue < 0.05
    hits_patch += stats.mannwhitneyu([np.median(null_nnd[i]) for i in a],
                                     [np.median(null_nnd[i]) for i in b]).pvalue < 0.05
print(f"fake experiments called 'significant' (should be ≈ 5 %):")
print(f"   every hexamer as a sample : {hits_hexamer / 2:.0f} %")
print(f"   one value per patch       : {hits_patch / 2:.0f} %")

🔍 **What just happened?** Treating thousands of hexamers as independent turns small, real differences *between patches*
into "significant" differences *between conditions* far more often than 5 %. Summarising per patch keeps the false-alarm
rate honest. In a real study the replication unit might be the tomogram, the cell, the grid or the culture — and the
paper's methods should tell you which one was used. *(We do not know which unit Jiang et al. used for each test; finding
out is part of reading the paper critically.)*

✏️ **Your turn.** Set `patch_sd=0.0` in `membrane_patch` (no patch-to-patch variability) and re-run 2.3b. What happens to
the per-hexamer false-alarm rate, and why?

### 2.4 Clusters: a rule, a scan and a model

**The rule.** "At least four hexamers at about 160–170 Å spacing" can be implemented with **single-linkage clustering**:
connect any two hexamers closer than a cutoff $d_c$; each connected group with ≥ 4 members is a cluster. We use
$d_c=20$ nm, a little above the spacing, to tolerate jitter.

**The scan.** Any fixed cutoff is a choice. Scanning $d_c$ and plotting the fraction of hexamers that end up in clusters
shows how the conclusion depends on that choice.

In [ ]:
# ── 2.4a Single-linkage clusters: fixed rule and cutoff scan ──
def clusters_at(points, cutoff, min_size=4):
    """Single-linkage clusters of points closer than `cutoff`; returns labels (-1 = not in a cluster of ≥ min_size)."""
    lab = fcluster(linkage(points, "single"), cutoff, "distance")
    ids, counts = np.unique(lab, return_counts=True)
    big = ids[counts >= min_size]
    return np.where(np.isin(lab, big), lab, -1)


cutoffs = np.arange(10, 61, 2)
half_cut = {}
fig, ax = plt.subplots(figsize=(6.4, 3.2))
for c, ps in patches.items():
    frac = np.array([[np.mean(clusters_at(p, d) >= 0) for d in cutoffs] for p, _ in ps])     # patches × cutoffs
    ax.plot(cutoffs, frac.mean(0), label=c)
    ax.fill_between(cutoffs, frac.min(0), frac.max(0), alpha=0.2)
    half_cut[c] = np.interp(0.5, frac.mean(0), cutoffs)
    print(f"{c:18s} fraction clustered at the fixed 20 nm rule: {frac[:, cutoffs == 20].mean():.2f};"
          f"  cutoff that clusters half of the hexamers: {half_cut[c]:.1f} nm")
ax.axvline(20, color="k", ls="--", lw=1); ax.text(21, 0.05, "fixed rule (20 nm)", fontsize=8)
ax.set_xlabel("single-linkage cutoff $d_c$ (nm)"); ax.set_ylabel("fraction of hexamers in clusters ≥ 4")
ax.legend(fontsize=8); plt.show()
print(f"ratio of half-clustering cutoffs, caspofungin / WT: {half_cut['WT + caspofungin'] / half_cut['WT']:.2f}")

🔍 **What just happened?** With the fixed 20 nm rule, the drug-treated clusters seem to *vanish*. The scan tells a more
accurate story: the clusters are still there, but everything is spaced further apart — the whole curve shifts right by
about the factor we built in. "Clusters disappear" and "clusters loosen" are different biological statements; the
analysis must be able to tell them apart.

**Cluster size and spacing.** To measure clusters in every condition we let the cutoff follow the data: $d_c$ = 1.3 ×
the median NND of each patch. For each cluster we record its **radius** (root-mean-square distance of its hexamers to
its centre) and its **intra-cluster distance** (median NND inside the cluster). We report the treated / WT ratio of the
per-patch medians with a **bootstrap 95 % confidence interval**: resample patches with replacement many times and see how
much the ratio moves.

In [ ]:
# ── 2.4b Cluster radius and intra-cluster spacing, with bootstrap confidence intervals ──
def cluster_measures(points):
    """Per-patch median cluster radius and intra-cluster NND (nm), using an adaptive single-linkage cutoff."""
    lab = clusters_at(points, 1.3 * np.median(nnd(points)))
    radii, spacing = [], []
    for k in np.unique(lab[lab >= 0]):
        q = points[lab == k]
        radii.append(np.sqrt(((q - q.mean(0)) ** 2).sum(1).mean()))
        spacing.append(np.median(nnd(q)))
    return np.median(radii), np.median(spacing)


measures = {c: np.array([cluster_measures(p) for p, _ in ps]) for c, ps in patches.items()}   # patches × 2


def boot_ratio(x, y, n_boot=2000):
    """Ratio of means mean(y)/mean(x) and its 95 % bootstrap interval (resampling patches)."""
    r = [rng.choice(y, len(y)).mean() / rng.choice(x, len(x)).mean() for _ in range(n_boot)]
    return y.mean() / x.mean(), np.percentile(r, [2.5, 97.5])


for j, what in enumerate(["cluster radius", "intra-cluster distance"]):
    print(f"{what}:  WT median {measures['WT'][:, j].mean():.1f} nm")
    for c in ["WT + caspofungin", "Δfen1"]:
        ratio, ci = boot_ratio(measures["WT"][:, j], measures[c][:, j])
        print(f"   {c:18s} / WT = {ratio:.2f}  (95 % CI {ci[0]:.2f}–{ci[1]:.2f}; built-in factor {CONDITIONS[c]})")

**A model-based check.** Is there really a separate population of tightly packed hexamers? A Gaussian mixture model
(Module 04) fitted to the **logarithm of the NND** answers this without any distance cutoff: if the hexamers fall into a
"clustered" and an "isolated" population, a two-component mixture should describe the data better than one, as judged by
the Bayesian information criterion (BIC). The weight of the low-distance component estimates the fraction of clustered
hexamers, and its mean estimates the spacing inside clusters. *(This particular check is our design for the toy; the
paper reports GMM AIC/BIC and k-means checks of its clusters without our having verified the details.)*

In [ ]:
# ── 2.4c Gaussian mixture on log nearest-neighbour distances, K chosen by BIC ──
for c, ps in patches.items():
    chosen, frac, mode = [], [], []
    for p, _ in ps:
        x = np.log(nnd(p))[:, None]
        fits = [GaussianMixture(k, random_state=SEED, n_init=3).fit(x) for k in (1, 2, 3, 4)]
        chosen.append(1 + int(np.argmin([f.bic(x) for f in fits])))
        low = int(np.argmin(fits[1].means_.ravel()))               # the tight ("clustered") component of K = 2
        frac.append(fits[1].weights_[low]); mode.append(np.exp(fits[1].means_.ravel()[low]))
    print(f"{c:18s} BIC-chosen K per patch: {chosen} | clustered fraction {np.mean(frac):.2f} "
          f"| typical spacing in clusters {np.mean(mode):.1f} nm")

🔍 **What just happened?** In most patches of every condition BIC prefers two populations (a few patches get three or
four, which split a population further), and about 80 % of hexamers belong to the tight one — exactly what we simulated. So the treated clusters did **not** dissolve, as the fixed 20 nm
rule suggested; they loosened, and the mixture's spacing estimate moves by about the built-in factor. A model check is
useful when it agrees with the other analyses and informative when it doesn't — it is not ground truth. *(Fitting
mixtures directly to the 2D coordinates is much less stable here: lattice-like clusters are poorly described by Gaussian
blobs. Try it in the Bonus.)*

### 2.5 Putting the case study together

A quick calculation with two densities reported in the paper: GS monomers at about 1,415 per µm² and Pma1 hexamers at
about 264 per µm².

In [ ]:
# ── 2.5 Do the tomograms and the proteomics agree on stoichiometry? ──
gs_per_um2, pma1_per_um2 = 1415, 264
print(f"GS : Pma1 hexamer ratio seen in tomograms ≈ {gs_per_um2 / pma1_per_um2:.1f} : 1   (paper: ≈ 5.4 : 1)")
print("ratio from mass-spectrometry proteomics of the membranes ≈ 3.3 : 1   (paper)")

🔍 **What just happened?** The tomograms show relatively *more* GS per Pma1 than the proteomics of the whole membrane
preparation. The paper reads this as GS being **enriched in the Pma1-rich microdomains** that were imaged — an example of
combining two methods (rung 4) to reach a conclusion neither could reach alone.

**Grading the paper's main claims on the ladder of Section 1:**

| Claim | Rung | Evidence |
|---|---|---|
| Pma1 hexamers pack into dense clusters; GS sits in Pma1-rich domains | 1–2 | annotation + spatial statistics; GS:Pma1 ratio vs proteomics |
| Caspofungin disperses Pma1 clusters (radius, spacing ≈ 1.7× in WT) | 3 | drug perturbation, statistics across tomograms |
| Caspofungin lowers membrane tension (≈ 2.6× in WT) | 4 | micropipette aspiration — independent of cryo-ET |
| Δ*fen1* membranes are already dispersed, tenser and less drug-susceptible | 3–4 | genetic perturbation + tension + susceptibility assays |
| The drug acts partly through lipids that reorganise the membrane | 5 | a proposed model consistent with all of the above |

Note what computation alone could **not** show: that dispersal *causes* anything, or that it matters for killing the
fungus. The orthogonal experiments carry those steps.

---
## 3 · Fitting components into a map

📍 **Where we are.** Section 2 started from coordinates. Often the starting point is a **map** of a whole assembly at low
resolution (say 20–30 Å) plus high-resolution structures of its parts. Which part sits where? This is the problem of
**fitting**, and one of the Xu lab's earliest papers (Zhang, Vasishtan, Xu, Topf & Alber 2010) is about doing it for
several components **simultaneously**.

Our toy assembly: one ribosome (4V4R) and **two copies** of RuBisCO (1BXN) touching it — *not* a real complex, just
zoo molecules in contact — at 30 Å resolution with noise.

In [ ]:
# ── 3.1 A toy assembly map ──
comp = {"ribosome": prepare_map(fetch_zoo_map("4v4r")[0], size=32), "RuBisCO": prepare_map(fetch_zoo_map("1bxn")[0], size=32)}
B = 56                                                             # assembly box (voxels of 10 Å)


def place(vol, R, centre):
    """Rotate a 32³ component by R and add it to an empty B³ box centred at integer voxel `centre` (z, y, x)."""
    out = np.zeros((B, B, B), np.float32)
    r = rotate_volume(vol, R)
    c = np.round(centre).astype(int) - 16
    dst = tuple(slice(max(0, c[i]), min(B, c[i] + 32)) for i in range(3))
    src = tuple(slice(max(0, -c[i]), 32 - max(0, c[i] + 32 - B)) for i in range(3))
    out[dst] += r[src]
    return out


fit_rng = np.random.default_rng(0)
truth = [("ribosome", random_rotation(fit_rng), (28, 26, 22)),
         ("RuBisCO", random_rotation(fit_rng), (28, 28, 40)),
         ("RuBisCO", random_rotation(fit_rng), (28, 12, 36))]
true_vols = [place(comp[n], R, c) for n, R, c in truth]
assembly = sum(true_vols)
target = lowpass(assembly, cutoff=1 / 3)                           # 10 Å voxels → 30 Å resolution
target = target + fit_rng.normal(0, 0.2 * target.std(), target.shape)
fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
axes[0].imshow(assembly.sum(0)); axes[0].set_title("true assembly (projection)")
axes[1].imshow(target.sum(0)); axes[1].set_title("map to interpret (30 Å, noisy)")
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

### 3.2 Exhaustive fitting, one component at a time

The standard approach (Module 11's search, now inside a map): for many rotations $R$, compute the cross-correlation of
the rotated component with the map for **every** translation at once by FFT, and keep the best peaks:

$$\mathrm{CC}(R,\mathbf t)=\sum_{\mathbf x}\big(\rho_{map}(\mathbf x)-\bar\rho_{map}\big)\,\rho_{comp}\big(R^{-1}(\mathbf x-\mathbf t)\big).$$

We keep the top candidate placements of each component type (best rotation per location).

In [ ]:
# ── 3.2 Candidate placements by rotational + FFT translational search ──
MAP_FT = np.fft.rfftn(target - target.mean())


def candidate_placements(vol, n_rot=300, per_rotation=3, keep=30, seed=1):
    """Return up to `keep` (score, R, centre) candidates, best first, at distinct locations."""
    r2, cand = np.random.default_rng(seed), []
    for _ in range(n_rot):
        R = random_rotation(r2)
        pad = np.zeros((B, B, B), np.float32); pad[:32, :32, :32] = rotate_volume(vol, R)
        cc = np.fft.irfftn(MAP_FT * np.conj(np.fft.rfftn(pad)), s=(B, B, B), axes=(0, 1, 2))
        peaks = np.argwhere(cc == ndimage.maximum_filter(cc, size=5, mode="wrap"))
        for p in peaks[np.argsort(-cc[tuple(peaks.T)])][:per_rotation]:
            cand.append((float(cc[tuple(p)]), R, (p + 16) % B))
    cand.sort(key=lambda x: -x[0])
    out = []
    for c in cand:                                                  # one candidate per location
        if all(np.linalg.norm(c[2] - o[2]) > 3 for o in out):
            out.append(c)
        if len(out) == keep:
            break
    return out


t0 = time.time()
cands = {name: candidate_placements(vol) for name, vol in comp.items()}
print(f"searched 2 × 300 rotations × {B**3:,} translations in {time.time() - t0:.0f} s")


def describe(name, c):
    """Distance (Å) to the nearest true copy of this component, and correlation with it."""
    v = place(comp[name], c[1], c[2])
    errs = [(np.linalg.norm(np.array(c[2]) - np.array(t)) * 10, pearson(v, tv))
            for (n, _, t), tv in zip(truth, true_vols) if n == name]
    err, cc = min(errs)
    return f"{name:9s} at {tuple(int(x) for x in c[2])}: {err:5.0f} Å from the nearest true copy, correlation {cc:.2f}"


print("Independent fitting (each copy takes the best placement of its type):")
for name in ["ribosome", "RuBisCO", "RuBisCO"]:
    print("  ", describe(name, cands[name][0]))

🔍 **What just happened?** The ribosome is placed correctly. But RuBisCO's best placement is **inside the ribosome**:
a small, dense object correlates best wherever there is most density. And the second copy, fitted independently, lands
in exactly the same place — independent fitting has no idea that two copies cannot occupy the same space.

### 3.3 Simultaneous fitting with a no-overlap constraint

Choose one candidate for every component **together**, maximising the total score subject to "no two placed components
overlap":

$$\max_{x}\ \sum_{c}\mathrm{CC}_c\,x_c\quad\text{subject to}\quad x_a+x_b\le 1\ \text{ if candidates } a,b \text{ clash},\qquad \sum_{c\in\text{type }k}x_c=n_k,\qquad x_c\in\{0,1\}.$$

*In words:* pick $n_k$ candidates of each type ($n_\text{ribosome}=1$, $n_\text{RuBisCO}=2$) with the highest total
correlation, never two that clash. This is a small **integer program**; with a few dozen candidates we can simply try
every combination.

In [ ]:
# ── 3.3 Clash matrix and exhaustive search over combinations ──
pool = [("ribosome", c) for c in cands["ribosome"]] + [("RuBisCO", c) for c in cands["RuBisCO"]]
masks = [place(comp[n], c[1], c[2]) > 0.2 for n, c in pool]
n_pool = len(pool)
clash = np.zeros((n_pool, n_pool), bool)
for a in range(n_pool):
    for b in range(a + 1, n_pool):
        overlap = (masks[a] & masks[b]).sum() / min(masks[a].sum(), masks[b].sum())
        clash[a, b] = clash[b, a] = overlap > 0.05                   # > 5 % of the smaller one overlaps
score = np.array([c[0] for _, c in pool])
ribo_idx = range(len(cands["ribosome"]))
rub_idx = range(len(cands["ribosome"]), n_pool)
best, n_tried = None, 0
for i in ribo_idx:
    for a, b in itertools.combinations(rub_idx, 2):
        n_tried += 1
        if not (clash[i, a] or clash[i, b] or clash[a, b]):
            total = score[i] + score[a] + score[b]
            if best is None or total > best[0]:
                best = (total, (i, a, b))
print(f"candidates: {len(cands['ribosome'])} ribosome, {len(cands['RuBisCO'])} RuBisCO → tried {n_tried:,} combinations")
print("Simultaneous fitting:")
for k in best[1]:
    print("  ", describe(*pool[k]))
fitted = sum(place(comp[pool[k][0]], pool[k][1][1], pool[k][1][2]) for k in best[1])
naive = sum(place(comp[n], cands[n][0][1], cands[n][0][2]) for n in ["ribosome", "RuBisCO", "RuBisCO"])
print(f"correlation of the fitted model with the true assembly: independent {pearson(naive, assembly):.2f},"
      f" simultaneous {pearson(fitted, assembly):.2f}")

In [ ]:
# ── 3.4 Compare the two answers ──
scenarios = {"truth": true_vols,
             "independent fitting": [place(comp[n], cands[n][0][1], cands[n][0][2]) for n in ["ribosome", "RuBisCO", "RuBisCO"]],
             "simultaneous fitting": [place(comp[pool[k][0]], pool[k][1][1], pool[k][1][2]) for k in best[1]]}
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5))
for ax, (title, vols) in zip(axes, scenarios.items()):
    ax.imshow(target.sum(0), cmap="gray")
    for v, colour, ls in zip(vols, ["tab:orange", "tab:cyan", "tab:pink"], ["-", "-", "--"]):
        ax.contour(v.sum(0), levels=[0.25 * v.sum(0).max()], colors=colour, linestyles=ls)
    ax.set_title(title); ax.axis("off")
plt.suptitle("orange: ribosome · cyan / pink dashed: the two RuBisCO copies", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Adding one constraint — components may not overlap — and solving for all components at once
recovers the assembly. Zhang *et al.* (2010) make this idea fast and general:

1. reduce every map to a few **feature points** (vector quantisation of the density at several thresholds, each point
   carrying a local density value);
2. match component points $i,k$ to assembly points $j,l$ with binary variables $x_{ij}$, rewarding similar local density
   $S$ and **agreement of internal distances** $G(a_{ik},b_{jl})$:

$$\max_X\ \sum_{i,j}S(u_{1i},u_{2j})\,x_{ij}+\sum_{i,k}\sum_{j,l}G(a_{ik},b_{jl})\,x_{ij}x_{kl},\qquad \sum_j x_{ij}\le1,\ \ \sum_i x_{ij}\le 1 ;$$

3. stack all components in one block-diagonal problem so that every assembly point is used at most once — that is what
   makes the fit **simultaneous** (off-diagonal blocks can even encode known contacts, e.g. from cross-linking);
4. solve the relaxed problem (an integer quadratic program, relaxed to a continuous one), generate an ensemble of 500
   solutions, rank them by cross-correlation, and refine with density-weighted iterative closest point.

The paper reports correct top-ranked solutions for 8 of 11 simulated assemblies; failures involved **pseudo-symmetric**
components — a reminder that symmetry makes fitting ambiguous.

✏️ **Your turn.** Fit the components **sequentially** instead: place the ribosome, subtract its density from the map, then
place one RuBisCO, subtract, place the other. Does this greedy strategy also work here? Can you design an assembly where
it fails but simultaneous fitting succeeds?

### 3.5 Beyond one rigid answer: conformations and integrative modelling

Real assemblies **move**. A proteasome gate opens, a ribosome ratchets, a chaperone cycles through states. Thalassinos,
Pandurangan, Xu, Alber & Topf (2013) is a *perspective* on how to describe such conformational states by combining
data types — **mass spectrometry** (which reports composition, stoichiometry and, through cross-links, distances) and
**cryo-EM/cryo-ET** (which reports shape), where heterogeneous subtomograms must be **aligned and classified** into
states (Modules 11–13, 20–21). No single method is enough; integrative modelling turns all of them into restraints on
one model. In our fitting toy, a cross-link saying "this RuBisCO touches the ribosome's small subunit" would be one more
constraint in the integer program — exactly what the off-diagonal blocks of Zhang *et al.* can encode.

---
## 4 · The proteome in situ: abundance, location and motion

Beck, Topf, Frazier, Tjong, Xu, Zhang & Alber (2011) review three ways of seeing a cell's proteome in space and time:
**fitting** structures into maps (Section 3), **visual proteomics** — finding complexes in tomograms (Modules 10–14) —
and **simulating** their motion in the crowded cytoplasm. Here we take a picked tomogram to three kinds of numbers.

### 4.1 From counts to concentrations

Biochemists think in molar concentrations. A count $N$ in a volume $V$ converts as

$$c=\frac{N}{N_A\,V},\qquad N_A=6.022\times10^{23}\ \text{mol}^{-1}.$$

In [ ]:
# ── 4.1 Counts → concentration ──
N_A = 6.022e23


def concentration_uM(count, volume_um3):
    """Molar concentration (µM) of `count` particles in `volume_um3` µm³ (1 µm³ = 1e-15 L)."""
    return count / (N_A * volume_um3 * 1e-15) * 1e6


RC, LC = 150.0, 900.0                                               # toy rod-shaped bacterium: radius and length (nm)
cell_volume = np.pi * (RC / 1000) ** 2 * (LC / 1000)                # µm³
for n in (100, 600, 2000):
    print(f"{n:5d} particles in our {cell_volume:.3f} µm³ cell → {concentration_uM(n, cell_volume):5.1f} µM")
print("(the review quotes ≈ 20 µM cytoplasmic ribosomes in Leptospira interrogans)")

### 4.2 Where are they? Spatial statistics against the right null model

We build an in silico bacterium with three species: **ribosomes**, mostly excluded from a central **nucleoid** and partly
strung into **polysomes** (chains along an mRNA, ≈ 28 nm apart); a "**proteasome-like**" complex placed uniformly; and
nothing else. Two questions a visual-proteomics study asks:

1. **Depletion.** Are ribosomes under-represented in the centre? Compare the observed fraction with a **null model** —
   the same number of particles placed uniformly in the cell — simulated many times (a Monte Carlo envelope).
2. **Association.** Do ribosomes sit at preferred distances from each other? The **pair-correlation function** $g(r)$
   compares the number of pairs at distance $r$ with the number expected under a null model: $g(r)>1$ means more pairs
   than expected.

In [ ]:
# ── 4.2a An in silico bacterium ──
RN, LN = 85.0, 550.0                                                # nucleoid radius and length (nm)


def in_cell(p):
    return (p[:, 1] ** 2 + p[:, 2] ** 2 < RC ** 2) & (np.abs(p[:, 0]) < LC / 2)


def in_nucleoid(p):
    return (p[:, 1] ** 2 + p[:, 2] ** 2 < RN ** 2) & (np.abs(p[:, 0]) < LN / 2)


def cytoplasm(p):
    return in_cell(p) & ~in_nucleoid(p)


def uniform_in(rng, n, region):
    """n points uniform inside a region (rejection sampling in the cell's bounding box)."""
    out = np.zeros((0, 3))
    while len(out) < n:
        p = np.column_stack([rng.uniform(-LC / 2, LC / 2, 4 * n), rng.uniform(-RC, RC, 4 * n), rng.uniform(-RC, RC, 4 * n)])
        out = np.vstack([out, p[region(p)]])
    return out[:n]


def polysomes(rng, n_chains=50, length=5, step=28.0):
    """Ribosome chains: a persistent random walk with ~28 nm steps, kept only if fully in the cytoplasm."""
    chains = []
    for s in uniform_in(rng, n_chains, cytoplasm):
        d = rng.normal(size=3); d /= np.linalg.norm(d)
        chain = [s]
        for _ in range(length - 1):
            d = d + 0.6 * rng.normal(size=3); d /= np.linalg.norm(d)
            chain.append(chain[-1] + step * d)
        chain = np.array(chain)
        if cytoplasm(chain).all():
            chains.append(chain)
    return np.vstack(chains)


cell_rng = np.random.default_rng(5)
ribosomes = np.vstack([uniform_in(cell_rng, 450, cytoplasm), uniform_in(cell_rng, 25, in_nucleoid),
                       polysomes(cell_rng)])
proteasomes = uniform_in(cell_rng, 300, in_cell)
print(f"{len(ribosomes)} ribosomes ({concentration_uM(len(ribosomes), cell_volume):.0f} µM), "
      f"{len(proteasomes)} proteasome-like complexes ({concentration_uM(len(proteasomes), cell_volume):.0f} µM)")
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.scatter(ribosomes[:, 0], ribosomes[:, 1], s=3, label="ribosomes")
ax.scatter(proteasomes[:, 0], proteasomes[:, 1], s=3, label="proteasome-like")
ax.add_patch(plt.Rectangle((-LN / 2, -RN), LN, 2 * RN, fill=False, ls="--", color="k"))
ax.set_aspect("equal"); ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)"); ax.legend(fontsize=8, loc="upper right")
ax.set_title("in silico bacterium, projected along z (dashed: nucleoid)"); plt.show()

In [ ]:
# ── 4.2b Depletion test with a Monte Carlo envelope ──
for name, pts in [("ribosomes", ribosomes), ("proteasome-like", proteasomes)]:
    observed = in_nucleoid(pts).mean()
    null = np.array([in_nucleoid(uniform_in(cell_rng, len(pts), in_cell)).mean() for _ in range(200)])
    p_low = (np.sum(null <= observed) + 1) / (len(null) + 1)
    print(f"{name:16s} fraction in the central region: {observed:.3f}  | uniform null: {null.mean():.3f} "
          f"(95 % of simulations {np.percentile(null, 2.5):.3f}–{np.percentile(null, 97.5):.3f}), one-sided p ≈ {p_low:.3f}")

In [ ]:
# ── 4.2c Pair-correlation function with two different null models ──
edges = np.arange(0, 161, 8.0)
centres_r = (edges[1:] + edges[:-1]) / 2


def pair_counts(points):
    """Number of point pairs in each distance shell (counts each unordered pair twice)."""
    tree = cKDTree(points)
    return np.diff(tree.count_neighbors(tree, edges))


observed = pair_counts(ribosomes)
null_cell = np.mean([pair_counts(uniform_in(cell_rng, len(ribosomes), in_cell)) for _ in range(20)], 0)
null_cyto = np.mean([pair_counts(uniform_in(cell_rng, len(ribosomes), cytoplasm)) for _ in range(20)], 0)
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(centres_r[1:], observed[1:] / null_cell[1:], "o-", label="null: uniform in the whole cell")
ax.plot(centres_r[1:], observed[1:] / null_cyto[1:], "s-", label="null: uniform outside the nucleoid")
ax.axhline(1, color="k", lw=0.8); ax.axvline(28, color="gold", lw=6, alpha=0.4, label="polysome spacing (28 nm)")
ax.set_xlabel("distance r (nm)"); ax.set_ylabel("g(r)"); ax.legend(fontsize=8); ax.set_title("ribosome–ribosome pair correlation")
plt.show()

🔍 **What just happened?** The depletion test clearly separates ribosomes (far fewer in the centre than the null allows)
from the uniformly placed complexes. The pair-correlation function shows the polysome peak near 28 nm — but its baseline
depends on the **null model**. Against "uniform in the whole cell", $g(r)$ stays above 1 over a wide range of short
distances simply because ribosomes are crowded into a smaller volume; against "uniform outside the nucleoid" it settles
near 1, isolating the true polysome signal. **Inhomogeneity can masquerade as clustering**; choosing the null model is a
scientific decision.

✏️ **Your turn.** Our simulated ribosomes may overlap; real ones cannot. Give both the simulation and the null model a
hard core (reject a new point closer than 20 nm to an existing one) and recompute $g(r)$. What happens at $r<20$ nm, and
why must a null model for real data include excluded volume?

### 4.3 How do they move? Diffusion in a crowded cytoplasm

Up to 30–50 % of the cytoplasm's volume is occupied by macromolecules (numbers quoted in the Beck review). Crowding slows
diffusion and so changes how often molecules meet. We simulate **hard discs** taking random steps (a Monte Carlo
version of Brownian motion; a move is rejected if it would create an overlap) and measure the **mean-squared displacement**

$$\mathrm{MSD}(t)=\big\langle\lVert\mathbf r(t)-\mathbf r(0)\rVert^2\big\rangle = 4Dt\quad\text{(in 2D, at long times)},$$

whose slope gives the diffusion coefficient $D$. We compare it with $D_0$, the value for a free particle.

In [ ]:
# ── 4.3 Hard-disc Monte Carlo at three area fractions ──
def crowded_diffusion(phi, rng, n=400, sweeps=500, sigma=0.15, radius=1.0, p_move=0.3):
    """Return D/D0 and the MSD curve for n hard discs at area fraction phi (periodic box)."""
    box = np.sqrt(n * np.pi * radius ** 2 / phi)
    side = int(np.ceil(np.sqrt(n * 2 / np.sqrt(3))))
    a = box / side
    i, j = np.meshgrid(np.arange(side), np.arange(side + 2))
    pos = np.stack([(i + 0.5 * (j % 2)) * a, j * np.sqrt(3) / 2 * a], -1).reshape(-1, 2)
    pos = pos[pos[:, 1] < box][:n]                                  # start from a non-overlapping hexagonal lattice
    unwrapped, msd = pos.copy(), []
    for s in range(sweeps + 100):
        moving = rng.random(n) < p_move
        step = np.where(moving[:, None], rng.normal(0, sigma, (n, 2)), 0.0)
        new = (pos + step) % box
        while True:                                                 # undo moves that create overlaps
            pairs = np.array(list(cKDTree(new, boxsize=box).query_pairs(2 * radius)))
            if len(pairs) == 0:
                break
            bad = np.zeros(n, bool); bad[pairs.ravel()] = True; bad &= moving
            if not bad.any():
                break
            new[bad], step[bad], moving = pos[bad], 0.0, moving & ~bad
        pos, unwrapped = new, unwrapped + step
        if s == 99:
            origin = unwrapped.copy()                               # 100 sweeps of equilibration first
        if s >= 100:
            msd.append(((unwrapped - origin) ** 2).sum(1).mean())
    t = np.arange(1, sweeps + 1)
    D = np.polyfit(t[sweeps // 2:], np.array(msd)[sweeps // 2:], 1)[0] / 4
    D0 = p_move * sigma ** 2 / 2                                    # free-particle value for these moves
    return D / D0, np.array(msd)


t0 = time.time()
diff_rng = np.random.default_rng(7)
fig, ax = plt.subplots(figsize=(6.4, 3.2))
for phi in (0.1, 0.3, 0.5):
    ratio, msd = crowded_diffusion(phi, diff_rng)
    ax.plot(msd, label=f"occupancy {phi:.0%}: D/D0 = {ratio:.2f}")
ax.set_xlabel("Monte Carlo sweeps"); ax.set_ylabel("MSD (disc radii²)"); ax.legend(fontsize=8)
ax.set_title("crowding slows diffusion"); plt.show()
print(f"(three simulations in {time.time() - t0:.1f} s)")

🔍 **What just happened?** At 50 % occupancy a disc diffuses several times slower than when free, because most attempted
moves collide with neighbours. Real cytoplasm adds sticky interactions, very different sizes and 3D geometry, which is
why realistic crowded simulations (Modules 08 and 12; Pei *et al.* 2016, Liu *et al.* 2020) and measured particle
positions from tomograms are so valuable for whole-cell models.

---
## 5 · What the lab's pattern mining found

Modules 14 and 20 taught the *methods* of template-free discovery. Here are the *biological* observations reported, and
the rung of the ladder each stands on.

| Paper | System | Reported observation | Rung | What would strengthen it |
|---|---|---|---|---|
| Xu *et al.* 2019, *Structure* (MPP; arXiv v3 numbers) | *H. gracilis*, *A. longum*, *B. bacteriovorus* | 10–15 recurring patterns per species, including membrane-associated ones; ribosome-like patterns (802 copies in *H. gracilis*) **depleted from a central, nucleoid-like region** in *B. bacteriovorus*; a GroEL-like pattern (481 instances) in *A. longum* | 1–2 | identity confirmation by higher resolution or labelling; a depletion test against a null model (Section 4.2) |
| Zeng *et al.* 2023, *PNAS* (DISCA) | *M. pneumoniae* (EMPIAR-10499) and four more datasets | without templates or labels: ribosome average at 14.17 Å; ribosome F1 0.893 vs 0.766 for raw template matching; groups matching TRiC, capped proteasome and phycobilisome arrays; cellular features (membranes, calcium-phosphate precipitates) | 1–2 | assigning identities independently of the averages; perturbations that change the patterns |

Pattern mining is a **hypothesis generator**: it tells you what is abundant and where. Turning "a ribosome-like pattern
is depleted from the centre" into biology needs the later rungs — just as the fungal-membrane study needed drugs,
mutants and tension measurements.

---
### 📄 Paper connection — fungal plasma membrane (Jiang *et al.*, Nature Communications 2025)

**Jiang J, Keniya MV, Puri A, Zhan X, Cheng J, Wang H, Lin G, Lee Y-K, Jaber N, Zhao C, Pang C, Hassoun Y, Zheng H,
Shor E, Shi Z, Lee S-H, Xu M, Perlin DS, Dai W.** *Molecular landscape of the fungal plasma membrane and implications for
antifungal action.* Nature Communications 16:9125 (2025). [doi:10.1038/s41467-025-64171-x](https://doi.org/10.1038/s41467-025-64171-x).
- **Question:** how are large complexes organised in the native *C. glabrata* plasma membrane, and how does caspofungin
  change that organisation?
- **Data:** cryo-ET of isolated plasma membranes (≈ 100 tomograms, 14 analysed) and of FIB-milled lamellae; deposited as
  EMD-45105 (untreated) and EMD-45106 (caspofungin-treated); proteomics MassIVE MSV000098079.
- **Methods:** EMAN2 CNN annotation (reported F1: ribosomes 0.788, Pma1 0.498) with manual curation; subtomogram averaging
  of GS; nearest-neighbour and cluster analysis (≥ 4 hexamers at 160–170 Å spacing; GMM AIC/BIC and k-means checks);
  Kruskal–Wallis and Dunn tests; proteomics; micropipette aspiration; drug-susceptibility assays.
- **Results reported:** Pma1 hexamers in dense clusters with GS enriched in them (GS:Pma1 ≈ 5.4:1 in tomograms vs ≈ 3.3:1
  by proteomics); caspofungin increases intra-cluster distance and cluster radius ≈ 1.7× (WT) and lowers membrane tension
  ≈ 2.6× (WT); Δ*fen1* membranes are already dispersed and less drug-susceptible.
- **What our toy keeps / leaves out:** keeps the measurements and statistics on point patterns; leaves out the imaging,
  annotation and averaging, and the real data (all our patterns are simulated).
- **The Xu lab's role:** M.X. and X.Z. (CMU) are co-authors; the paper does not itemise which computational analysis each
  author performed, so we do not attribute specific steps to the lab.

### 📄 Paper connection — simultaneous fitting (Zhang, Vasishtan, Xu, Topf & Alber, Bioinformatics 2010)

**Zhang S, Vasishtan D, Xu M, Topf M, Alber F.** *A fast mathematical programming procedure for simultaneous fitting of
assembly components into cryoEM density maps.* Bioinformatics 26(12):i261–i268 (ISMB 2010).
[doi:10.1093/bioinformatics/btq201](https://doi.org/10.1093/bioinformatics/btq201).
- **What the paper does:** feature points by vector quantisation; point matching as an integer quadratic program over all
  components at once, solved by relaxation; ensemble of 500 solutions ranked by cross-correlation; density-weighted ICP.
- **Results reported:** 11 simulated assemblies (20 Å): lowest-RMSD solutions 1.3–6.5 Å, refined to 0.4–1.8 Å; top-ranked
  solution correct in 8/11; GroEL ring from EMD-1046 (23.5 Å): lowest RMSD 8.6 Å; 0.1–2.6 s per run.
- **What our toy keeps / leaves out:** keeps simultaneous placement under a shared constraint; replaces feature points and
  the IQP with exhaustive candidate search and brute-force combination.
- **Code:** Matlab package available on request (no public repository).

### 📄 Paper connection — the cell's proteome in space and time (Beck *et al.*, J. Struct. Biol. 2011)

**Beck M, Topf M, Frazier Z, Tjong H, Xu M, Zhang S, Alber F.** *Exploring the spatial and temporal organization of a
cell's proteome.* J Struct Biol 173(3):483–496 (2011). [doi:10.1016/j.jsb.2010.11.011](https://doi.org/10.1016/j.jsb.2010.11.011).
A **review**: density fitting (rigid, flexible, simultaneous), visual proteomics by template matching in tomograms
(*Leptospira interrogans*, *Mycoplasma pneumoniae*), and simulation of diffusion and reactions in crowded cytoplasm.

### 📄 Paper connection — conformational states (Thalassinos *et al.*, Structure 2013)

**Thalassinos K, Pandurangan AP, Xu M, Alber F, Topf M.** *Conformational states of macromolecular assemblies explored by
integrative structure calculation.* Structure 21(9):1500–1508 (2013). [doi:10.1016/j.str.2013.08.006](https://doi.org/10.1016/j.str.2013.08.006).
A **perspective** on computational methods for mass spectrometry and for cryo-EM/cryo-ET (with a focus on aligning and
classifying heterogeneous subtomograms) and on integrating them to describe assembly dynamics. ⚠️ We verified the
abstract only; read the paper for its specific examples.

### 📄 Biological findings — MPP (Xu *et al.*, Structure 2019) and DISCA (Zeng *et al.*, PNAS 2023)

See Modules 14 and 20 for the methods and full citations (doi:10.1016/j.str.2019.01.005; doi:10.1073/pnas.2213149120).

## ✅ Checkpoint

1. Why can a per-hexamer test give a tiny p-value when there is no treatment effect?
   <details><summary>Answer</summary>Hexamers in one patch share that patch's conditions, so they are not independent. Pooling them treats patch-to-patch variation as if it were a treatment difference and hugely overstates the sample size (pseudo-replication).</details>
2. With a fixed 20 nm cluster rule, drug-treated clusters "disappear". Why is the cutoff scan more informative?
   <details><summary>Answer</summary>It shows the clusters still exist at larger cutoffs: the whole curve shifts by the spacing expansion. "Loosened" and "dissolved" are different claims, and a single cutoff cannot distinguish them.</details>
3. Why does RuBisCO fit inside the ribosome when fitted alone?
   <details><summary>Answer</summary>Unnormalised cross-correlation of a small dense object is highest wherever the map has the most density; without knowing that space is taken, the search puts it in the ribosome.</details>
4. What makes the fitting in Zhang et al. 2010 "simultaneous"?
   <details><summary>Answer</summary>All components' feature points are matched in one optimisation in which every assembly point can be used at most once, so components cannot claim the same density.</details>
5. Why does $g(r)$ depend on the null model?
   <details><summary>Answer</summary>g(r) is observed pairs divided by expected pairs. If the null spreads points over a larger volume than they really occupy, expected pairs are too few and g(r) looks inflated at all short distances — inhomogeneity looks like clustering.</details>
6. Which rung of the ladder is "caspofungin lowers membrane tension", and why does it matter for the cryo-ET result?
   <details><summary>Answer</summary>Rung 4: an orthogonal measurement. It independently supports the idea that the drug changes the membrane, making the lipid-mediated model more credible than the cryo-ET clustering result alone.</details>

## 🧠 Questions & discussion

1. The EMAN2 annotation of Pma1 had a reported F1 of about 0.5 before curation. How could missed or false hexamers bias
   nearest-neighbour distances and cluster statistics? How would you test the robustness of the conclusion?
2. Isolated membranes are not intact cells. What could change during isolation, and which experiment in the paper
   partly addresses this?
3. The GS:Pma1 ratio differs between tomograms and proteomics. Give two explanations other than enrichment in Pma1
   domains.
4. In the fitting toy, what would happen if the map were at 60 Å resolution? Try it.
5. MPP reports ribosome-like patterns depleted from a central region. Design the null model you would use to test it,
   and say what could make the test misleading.

## 🚀 Bonus — going further

- **Real data.** Download EMD-45105 and EMD-45106 from the EMDB and look at the membranes in ChimeraX or napari; compare
  with the toy patches.
- **2D mixtures.** Fit `GaussianMixture` to the 2D hexamer coordinates for K = 2…40 and plot BIC. Why does BIC keep
  asking for more components for lattice-like clusters?
- **Ripley's K.** Implement Ripley's K/L function for the membrane patches with edge correction and compare it with the
  NND analysis.
- **The IQP.** Implement Zhang *et al.*'s point matching on a 2D toy: k-means feature points, the quadratic objective,
  a spectral relaxation (leading eigenvector of the affinity matrix) and Hungarian rounding.
- **Reactions.** Add a binding reaction to the hard-disc simulation and measure how the encounter rate changes with
  crowding.

## 📝 Lab note

```text
Module 25 — date:
Input: simulated membrane point patterns (3 conditions × 7 patches); toy assembly map; in silico bacterium; hard discs
Output: NND statistics, cluster ratios with CIs, simultaneous fit, concentrations, depletion test, g(r), D/D0
Method: Kruskal–Wallis/Dunn, single linkage, bootstrap, GMM-BIC; FFT fitting + integer program; Monte Carlo nulls
Unit of replication I chose and why:
Null model I chose for g(r) and why:
A claim and its rung on the ladder of evidence:
Paper link: Jiang et al. 2025; Zhang et al. 2010; Beck et al. 2011; Thalassinos et al. 2013
```

## 📚 References

1. Jiang J *et al.* Molecular landscape of the fungal plasma membrane and implications for antifungal action. *Nat Commun* 16:9125 (2025). doi:10.1038/s41467-025-64171-x
2. Zhang S, Vasishtan D, Xu M, Topf M, Alber F. A fast mathematical programming procedure for simultaneous fitting of assembly components into cryoEM density maps. *Bioinformatics* 26:i261–i268 (2010). doi:10.1093/bioinformatics/btq201
3. Beck M *et al.* Exploring the spatial and temporal organization of a cell's proteome. *J Struct Biol* 173:483–496 (2011). doi:10.1016/j.jsb.2010.11.011
4. Thalassinos K, Pandurangan AP, Xu M, Alber F, Topf M. Conformational states of macromolecular assemblies explored by integrative structure calculation. *Structure* 21:1500–1508 (2013). doi:10.1016/j.str.2013.08.006
5. Xu M *et al.* De novo structural pattern mining in cellular electron cryotomograms. *Structure* 27:679–691 (2019). doi:10.1016/j.str.2019.01.005
6. Zeng X *et al.* High-throughput cryo-ET structural pattern mining by unsupervised deep iterative subtomogram clustering. *PNAS* 120:e2213149120 (2023). doi:10.1073/pnas.2213149120
7. Lazic SE. The problem of pseudoreplication in neuroscientific studies: is it affecting your analysis? *BMC Neurosci* 11:5 (2010).
8. Dunn OJ. Multiple comparisons using rank sums. *Technometrics* 6:241–252 (1964).

Deep dives in this repository: [`SubtomogramAveraging`](../../SubtomogramAveraging/) ·
[`SubtomogramClassification`](../../SubtomogramClassification/) · [`TomogramSegmentation`](../../TomogramSegmentation/).

## ➡️ Next

**[26 · Capstone: an end-to-end visual-proteomics study and a paper audit](26_capstone_visual_proteomics.ipynb)** — put
every tool of the course together on a cell whose composition is hidden from you, then audit a paper like a reviewer.
Course overview: [`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).